In [1]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score

# Create dataset
X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_classes=2,
    random_state=42
)

# Create model
model = LogisticRegression(max_iter=1000)

# Create 5-fold CV
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [2]:
scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="accuracy"
)

print("Fold Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

Fold Scores: [0.88 0.85 0.92 0.9  0.9 ]
Mean Accuracy: 0.89
Standard Deviation: 0.023664319132398488


# ML Evaluation & Selection — 02. Cross-Validation

## 1. What is Cross-Validation?

Cross-Validation is a technique used to **evaluate how well a machine learning model generalizes to unseen data**.

Instead of relying on only one train-validation split, the data is divided into multiple parts and the model is evaluated multiple times.

```text
Single Split
    ↓
One validation score
    ↓
May depend heavily on that particular split
```

```text
Cross-Validation
    ↓
Multiple train/validation splits
    ↓
Multiple scores
    ↓
Mean + variation
    ↓
More reliable performance estimate
```

---

# 2. K-Fold Cross-Validation

The most common method is **K-Fold Cross-Validation**.

If:

```text
K = 5
```

the dataset is divided into **5 folds**.

```text
┌────────┬────────┬────────┬────────┬────────┐
│ Fold 1 │ Fold 2 │ Fold 3 │ Fold 4 │ Fold 5 │
└────────┴────────┴────────┴────────┴────────┘
```

The model is trained and validated **5 times**.

---

# 3. How K-Fold Works

### Fold 1

```text
Validation → Fold 1
Training   → Fold 2 + 3 + 4 + 5
```

### Fold 2

```text
Validation → Fold 2
Training   → Fold 1 + 3 + 4 + 5
```

### Fold 3

```text
Validation → Fold 3
Training   → Fold 1 + 2 + 4 + 5
```

### Fold 4

```text
Validation → Fold 4
Training   → Fold 1 + 2 + 3 + 5
```

### Fold 5

```text
Validation → Fold 5
Training   → Fold 1 + 2 + 3 + 4
```

Therefore, **every sample gets a chance to be part of the validation set**.

---

# 4. Cross-Validation Score

Suppose the five folds produce:

```text
Fold 1 → 91%
Fold 2 → 94%
Fold 3 → 89%
Fold 4 → 93%
Fold 5 → 92%
```

The mean CV score is:

$$
CV\ Score =
\frac{91 + 94 + 89 + 93 + 92}{5}
$$

$$
CV\ Score = 91.8\%
$$

The mean gives us the **average model performance across the folds**.

---

# 5. Standard Deviation

We should also look at the **standard deviation** of the fold scores.

It tells us how much the model's performance changes between different folds.

```text
Low Standard Deviation
→ Performance is consistent

High Standard Deviation
→ Performance varies significantly
```

Example:

```text
Model A:
[0.91, 0.92, 0.90, 0.91, 0.92]

→ Consistent performance
```

```text
Model B:
[0.99, 0.78, 0.95, 0.82, 0.97]

→ Performance varies significantly
```

So we generally consider:

```text
Mean CV Score
+
Standard Deviation
```

rather than looking only at one score.

---

# 6. Why Use Cross-Validation?

A single train-test split can produce a result that depends heavily on **which samples were selected**.

```text
Split 1 → 91%
Split 2 → 96%
Split 3 → 88%
```

Cross-Validation gives us multiple evaluations:

```text
Multiple Splits
      ↓
Multiple Scores
      ↓
Mean + Standard Deviation
      ↓
More reliable estimate
```

### Core idea

> **Cross-Validation helps us estimate model performance more reliably by evaluating the model on multiple different validation splits.**

---

# 7. Training Data vs Test Data

A proper ML workflow is:

```text
                 Full Dataset
                      ↓
             ┌────────┴────────┐
             ↓                 ↓
       Training Data        Test Data
             ↓
     Cross-Validation
             ↓
    Model Selection /
    Hyperparameter Tuning
             ↓
       Final Model
             ↓
     Evaluate on Test Set
```

The **test set should remain untouched** while performing Cross-Validation and model selection.

Why?

Because the test set is supposed to represent **truly unseen data**.

---

# 8. K-Fold Coding

```python
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score

# Create dataset
X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_classes=2,
    random_state=42
)

# Create model
model = LogisticRegression(max_iter=1000)

# Create 5-fold cross-validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Perform cross-validation
scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="accuracy"
)

print("Fold Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())
```

---

# 9. Understanding `cross_val_score()`

```python
scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="accuracy"
)
```

### `model`

The ML model being evaluated.

### `X`

Input features.

### `y`

Target values.

### `cv`

Defines the Cross-Validation strategy.

Here:

```python
cv=kf
```

means we are using our **5-fold KFold** object.

### `scoring`

Defines what metric should be calculated.

Example:

```python
scoring="accuracy"
```

---

# 10. Stratified K-Fold

For classification, we often use **StratifiedKFold**.

Its purpose is to maintain approximately the same **class distribution** in every fold.

Suppose the dataset contains:

```text
90% → Class 0
10% → Class 1
```

Stratified K-Fold tries to maintain this distribution:

```text
Fold 1 → 90% / 10%
Fold 2 → 90% / 10%
Fold 3 → 90% / 10%
Fold 4 → 90% / 10%
Fold 5 → 90% / 10%
```

### Coding

```python
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring="accuracy"
)

print("Scores:", scores)
print("Mean:", scores.mean())
print("Std:", scores.std())
```

### Remember

```text
Classification
→ StratifiedKFold is often preferred

Regression
→ KFold is commonly used
```

---

# 11. Choosing K

Common choices are:

```text
K = 5
K = 10
```

### K = 5

```text
→ Faster
→ Fewer training runs
```

### K = 10

```text
→ More evaluations
→ More computationally expensive
```

There is no universal perfect value.

**5-fold and 10-fold are common practical choices.**

---

# 12. Cross-Validation and Hyperparameter Tuning

Cross-Validation becomes especially important during **Hyperparameter Tuning**.

```text
Different Hyperparameters
          ↓
   Cross-Validation
          ↓
   Compare CV Scores
          ↓
 Choose Best Parameters
```

This is exactly what tools such as `GridSearchCV` use.

---

# 13. Final Mental Model

```text
                 Cross-Validation
                        ↓
                Split data into K folds
                        ↓
             ┌──────────┼──────────┐
             ↓          ↓          ↓
          Fold 1      Fold 2     Fold 3 ...
             ↓          ↓          ↓
           Score      Score      Score
             └──────────┼──────────┘
                        ↓
                 Mean CV Score
                        +
                 Standard Deviation
                        ↓
            Estimate model performance
```

---

# ⭐ Core Rules to Remember

```text
Single Train/Test Split
→ One performance estimate
```

```text
K-Fold Cross-Validation
→ Multiple performance estimates
```

```text
Mean CV Score
→ Average performance
```

```text
Standard Deviation
→ Performance consistency
```

```text
Classification
→ StratifiedKFold is often preferred
```

```text
Regression
→ KFold is commonly used
```

```text
Test Set
→ Keep untouched until final evaluation
```

### One-line definition

> **Cross-Validation repeatedly trains and validates a model on different parts of the training data to obtain a more reliable estimate of its generalization performance.**
